# Mind the Gap: BERT with 5-fold cross-validation on Google Colab

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.

Needs `project_upload.zip` (created on the Mac with `zip -r project_upload.zip src data/processed`)
in Google Drive, folder `MyDrive/mind-the-gap/`.

**Results are written directly to Google Drive** (`MyDrive/mind-the-gap/cv_bert/`), one folder per
fold and learning rate. If Colab disconnects or the GPU limit is reached, just run the notebook again
later: finished runs are skipped and training continues where it stopped.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - Runtime > Change runtime type > T4 GPU')

## 1. Project from Google Drive, results to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/mind-the-gap'

import os
!rm -rf /content/project && mkdir -p /content/project
!unzip -qo "$DRIVE_DIR/project_upload.zip" -d /content/project
os.chdir('/content/project')

# results/cv/bert -> Google Drive, so every finished run is saved immediately
!mkdir -p "$DRIVE_DIR/cv_bert" results/cv
!ln -sfn "$DRIVE_DIR/cv_bert" results/cv/bert
print(os.listdir('data/processed'))
!grep -c "def train_one" src/train_bert.py   # must print 1 (cross-validation version)

In [ ]:
# torchao preinstalled on Colab is too old for recent peft and is not needed
!pip uninstall -y -q torchao
!pip install -q peft

## 2. Smoke test (2-3 minutes)
Pretrained BERT, fold 0, 200 windows, 2 epochs. Written to a separate folder, not used in the results.

In [ ]:
!python -m src.train_bert --signal resp --folds 0 --lrs 2e-4 --subset 200 --epochs 2 --min-epochs 1 --out-name bert_smoke

## 3. Training: one cell per signal
Each cell trains 5 folds x 2 learning rates = 10 models (about 1.5 hours for HR and respiration,
1 hour for stress). Run the cells one at a time. If the session stops, run sections 1 and 3 again:
finished runs are skipped.

To share the work between two Colab accounts, use `--folds`, e.g. `--folds 0,1,2` on one account and
`--folds 3,4` on the other, then copy all fold folders into the same `cv_bert` folder.

In [ ]:
!python -m src.train_bert --signal hr

In [ ]:
!python -m src.train_bert --signal resp

In [ ]:
!python -m src.train_bert --signal stress

## 4. Check what is finished

In [ ]:
!find results/cv/bert -name run.json | sort

When all 30 runs are listed (3 signals x 5 folds x 2 learning rates), download the folder
`MyDrive/mind-the-gap/cv_bert` and copy its content into `results/cv/bert/` of the repo, then run
`python -m src.evaluate` on the Mac.